# fm4tag — Training Time Profiling

Interactive notebook to find time bottlenecks in pretraining and fine-tuning.

**Profiling strategy (coarse → fine):**
1. **Dataloader throughput** — isolate I/O vs compute; sweep `num_workers` / `prefetch_factor` / `pin_memory`
2. **Lightning SimpleProfiler** — wall-clock per hook; ~zero overhead, great first pass
3. **Lightning AdvancedProfiler** — cProfile per hook; function-level call graph
4. **Lightning PyTorchProfiler** — GPU + CPU op-level; writes Chrome trace for Perfetto
5. **Manual micro-benchmarks** — `torch.cuda.Event` timing of individual sub-components
6. **`torch.profiler` deep dive** — direct profiler with memory stats and full trace
7. **GPU memory audit** — peak and per-step allocation

Run sections **in order**; each builds on the previous.
Trace files (sections 4, 6) are written to `../outputs/profiling/`.

---
## 0 — Setup

In [1]:
import os
import sys
import time
import io
import pstats
import unittest.mock as mock
from pathlib import Path

import torch
import lightning as L
from omegaconf import OmegaConf

# ── project root on sys.path ─────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fm4tag.engine import _build_encoders, _build_profiler
from fm4tag.data import PT_FT_DataModule
from fm4tag.models import PretrainModule
from fm4tag.data.augmentations import embed_data

# ── reproducibility ───────────────────────────────────────────────────────────
torch.set_float32_matmul_precision('high')
L.seed_everything(42, workers=True)

print(f'PyTorch  : {torch.__version__}')
print(f'Lightning: {L.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE_STR = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DEVICE     = torch.device(DEVICE_STR)

Seed set to 42


PyTorch  : 2.10.0+cu128
Lightning: 2.6.1
CUDA available: True
GPU : NVIDIA A40
VRAM: 47.7 GB


In [2]:
CONFIG_PATH = '../src/fm4tag/configs/time_experiment.yaml'
assert os.path.exists(CONFIG_PATH), f'Config not found: {CONFIG_PATH}'

cfg = OmegaConf.load(CONFIG_PATH)


def make_cfg(**dotkey_overrides):
    """Return a fresh OmegaConf config with dot-key overrides applied.

    E.g.::

        make_cfg(**{'pretrain_dataloader.num_workers': 8,
                    'encoder.dim': 32})

    Keys are split on '.' and turned into a nested dict before merging,
    so they correctly overwrite nested config entries.
    Note: Lightning Trainer kwargs such as ``limit_train_batches`` must be
    passed directly to ``L.Trainer(...)`` — they are not part of cfg.trainer.
    """
    nested: dict = {}
    for dotkey, val in dotkey_overrides.items():
        parts = dotkey.split('.')
        d = nested
        for p in parts[:-1]:
            d = d.setdefault(p, {})
        d[parts[-1]] = val
    return OmegaConf.merge(cfg, OmegaConf.create(nested))


print(OmegaConf.to_yaml(cfg))

phase: pretrain
action: fit
seed: 42
experiment_name: model_0
output_dir: outputs
global_object: jets
constituent_objects:
- tracks
variables:
  jets:
    inputs:
    - pt_btagJes
    - eta_btagJes
    label: flavour_label
    unique_labels:
    - 0
    - 1
    - 2
    - 3
  tracks:
    inputs:
      continuous:
      - d0
      - z0SinTheta
      - dphi
      - deta
      - qOverP
      - lifetimeSignedD0Significance
      - lifetimeSignedZ0SinThetaSignificance
      - phiUncertainty
      - thetaUncertainty
      - qOverPUncertainty
      categorical:
      - numberOfPixelHits
      - numberOfSCTHits
      - numberOfInnermostPixelLayerHits
      - numberOfNextToInnermostPixelLayerHits
      - numberOfInnermostPixelLayerSharedHits
      - numberOfInnermostPixelLayerSplitHits
      - numberOfPixelSharedHits
      - numberOfPixelSplitHits
      - numberOfSCTSharedHits
      cat_classes:
        numberOfPixelHits:
        - 0
        - 1
        - 2
        - 3
        - 4
        - 5
  

In [3]:
# ── Shared helpers used in multiple sections ──────────────────────────────────

PROF_DIR = Path('../outputs/profiling')
PROF_DIR.mkdir(parents=True, exist_ok=True)


def to_device(batch: dict, device) -> dict:
    """Move a collated batch dict to *device* in-place and return it."""
    batch['label']  = batch['label'].to(device)
    batch['global'] = batch['global'].to(device)
    batch['constituents'] = {
        obj: {k: v.to(device) for k, v in feats.items()}
        for obj, feats in batch['constituents'].items()
    }
    return batch


def build_pretrain_module(c=None):
    """Convenience: build encoders + PretrainModule and move to DEVICE."""
    c = c or cfg
    enc = _build_encoders(c)
    mod = PretrainModule(enc, c).to(DEVICE_STR)
    return mod


def forward_loss(module, batch):
    """Call module._compute_loss() directly — no Lightning logging required."""
    loss, _ = module._compute_loss(batch)
    return loss


print('Helpers defined.')

Helpers defined.


---
## 1 — Dataloader Throughput

Isolate **I/O bottlenecks** before profiling training.  
A common mistake is attributing slow training to the model when the dataloader
is starving the GPU.

We measure:
* Baseline throughput (config defaults)
* `num_workers` sweep
* `prefetch_factor` sweep
* `pin_memory` toggle

In [4]:
def dataloader_throughput(dl, n_batches: int = 30, device=None, warmup: int = 3) -> float:
    """Measure samples/s for *n_batches* batches.

    Args:
        dl:       DataLoader to benchmark.
        n_batches: Number of batches to time (after warmup).
        device:   If given, move each batch to this device (measures PCIe + H→D).
        warmup:   Batches to discard before timing (avoids cold-start artefacts).
    """
    it = iter(dl)

    def _next():
        nonlocal it
        try:
            return next(it)
        except StopIteration:
            it = iter(dl)
            return next(it)

    for _ in range(warmup):
        _next()

    if device and torch.cuda.is_available():
        torch.cuda.synchronize(device)

    t0 = time.perf_counter()
    total_samples = 0
    for _ in range(n_batches):
        batch = _next()
        if device:
            to_device(batch, device)
            if torch.cuda.is_available():
                torch.cuda.synchronize(device)
        total_samples += batch['label'].shape[0]

    elapsed = time.perf_counter() - t0
    return total_samples / elapsed  # samples/s


N_DL_BATCHES = 30
print('dataloader_throughput defined.')

dataloader_throughput defined.


In [5]:
# ── Baseline (config defaults) ────────────────────────────────────────────────
dm_base = PT_FT_DataModule(cfg, phase='pretrain')
dm_base.setup('fit')
dl_base = dm_base.train_dataloader()

thr_cpu = dataloader_throughput(dl_base, N_DL_BATCHES, device=None)
thr_gpu = dataloader_throughput(dl_base, N_DL_BATCHES, device=DEVICE) if torch.cuda.is_available() else None

dl_cfg = cfg.pretrain_dataloader
print(f'Config: num_workers={dl_cfg.num_workers}, '
      f'prefetch={dl_cfg.prefetch_factor}, '
      f'pin_memory={dl_cfg.pin_memory}')
print(f'  CPU-only throughput : {thr_cpu:>10,.0f} samples/s')
if thr_gpu:
    print(f'  To-GPU throughput   : {thr_gpu:>10,.0f} samples/s')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


Config: num_workers=12, prefetch=2, pin_memory=True
  CPU-only throughput :      4,511 samples/s
  To-GPU throughput   :      4,499 samples/s


In [6]:
# ── num_workers sweep ─────────────────────────────────────────────────────────
# Rule of thumb: start at 4× num GPUs, then profile.
# When num_workers=0, prefetch_factor must be None (DataLoader requirement).

workers_to_test = [0, 2, 4, 8, 12, 16, 20, 22]
results_workers: dict[int, float] = {}

for nw in workers_to_test:
    pf = 2 if nw > 0 else None  # prefetch_factor only valid when num_workers > 0
    overrides = {'pretrain_dataloader.num_workers': nw}
    if pf is not None:
        overrides['pretrain_dataloader.prefetch_factor'] = pf
    c = make_cfg(**overrides)

    dm = PT_FT_DataModule(c, phase='pretrain')
    dm.setup('fit')
    dl = dm.train_dataloader()
    thr = dataloader_throughput(dl, N_DL_BATCHES, device=DEVICE)
    results_workers[nw] = thr
    print(f'  num_workers={nw:>2d}  →  {thr:>10,.0f} samples/s')

best_nw = max(results_workers, key=results_workers.get)
print(f'\nBest num_workers: {best_nw}  ({results_workers[best_nw]:,.0f} samples/s)')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


  num_workers= 0  →         434 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
  num_workers= 2  →         865 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
  num_workers= 4  →       1,500 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-f

In [7]:
# ── prefetch_factor sweep ─────────────────────────────────────────────────────
# Only meaningful when num_workers > 0.
pf_to_test = [1, 2, 4, 8] if best_nw > 0 else [None]
results_prefetch: dict = {}

for pf in pf_to_test:
    overrides = {'pretrain_dataloader.num_workers': best_nw}
    if pf is not None:
        overrides['pretrain_dataloader.prefetch_factor'] = pf
    c = make_cfg(**overrides)

    dm = PT_FT_DataModule(c, phase='pretrain')
    dm.setup('fit')
    dl = dm.train_dataloader()
    thr = dataloader_throughput(dl, N_DL_BATCHES, device=DEVICE)
    results_prefetch[pf] = thr
    print(f'  prefetch_factor={pf}  →  {thr:>10,.0f} samples/s')

best_pf = max(results_prefetch, key=results_prefetch.get)
print(f'\nBest prefetch_factor: {best_pf}  ({results_prefetch[best_pf]:,.0f} samples/s)')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


  prefetch_factor=1  →       9,733 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
  prefetch_factor=2  →       9,873 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
  prefetch_factor=4  →       7,506 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transform

In [8]:
# ── pin_memory toggle ─────────────────────────────────────────────────────────
for pm in [True, False]:
    overrides = {
        'pretrain_dataloader.num_workers': best_nw,
        'pretrain_dataloader.pin_memory':  pm,
    }
    if best_nw > 0 and best_pf is not None:
        overrides['pretrain_dataloader.prefetch_factor'] = best_pf
    c = make_cfg(**overrides)

    dm = PT_FT_DataModule(c, phase='pretrain')
    dm.setup('fit')
    dl = dm.train_dataloader()
    thr = dataloader_throughput(dl, N_DL_BATCHES, device=DEVICE)
    print(f'  pin_memory={pm!s:<5}  →  {thr:>10,.0f} samples/s')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


  pin_memory=True   →       7,486 samples/s

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
  pin_memory=False  →       8,657 samples/s


---
## 2 — Lightning SimpleProfiler

Wall-clock time **per Lightning hook** for one mini-epoch.  
Near-zero overhead — use this as the first pass to spot which hook dominates.

**Key hooks to watch:**
* `training_step` — forward + loss computation
* `on_train_batch_start` / `on_train_batch_end` — callback overhead
* `[fetch]_train_batch` — dataloader stall time (if high → I/O bottleneck)

In [9]:
from lightning.pytorch.profilers import SimpleProfiler
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import TQDMProgressBar

LIMIT_STEPS_SIMPLE = 200   # keep < 1 min

# We pass limit_train_batches directly to L.Trainer — it is a Trainer kwarg,
# not a config key, so it must NOT go through make_cfg.
cfg_simple = make_cfg()   # no cfg changes needed for this profiler run

cfg_simple_dict = OmegaConf.to_container(cfg_simple.trainer, resolve=True)
cfg_simple_dict['strategy'] = 'auto'

In [10]:
from lightning.pytorch.profilers import SimpleProfiler
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import TQDMProgressBar

LIMIT_STEPS_SIMPLE = 200   # keep < 1 min

# We pass limit_train_batches directly to L.Trainer — it is a Trainer kwarg,
# not a config key, so it must NOT go through make_cfg.
cfg_simple = make_cfg()   # no cfg changes needed for this profiler run

simple_profiler = SimpleProfiler(
    dirpath=str(PROF_DIR), filename='simple', extended=True
)

trainer_simple = L.Trainer(
    **cfg_simple_dict,
    limit_train_batches=LIMIT_STEPS_SIMPLE,
    limit_val_batches=10,
    callbacks=[TQDMProgressBar(refresh_rate=50)],
    logger=CSVLogger('../outputs', name='profiling_simple'),
    profiler=simple_profiler,
    enable_model_summary=False,
    enable_checkpointing=False,
)

module_simple = build_pretrain_module(cfg_simple)
dm_simple = PT_FT_DataModule(cfg_simple, phase='pretrain')

trainer_simple.fit(module_simple, dm_simple)
print(simple_profiler.summary())

/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /storage3/DSIP/rriva/research/fm4tag/.venv/lib/pytho ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
Loading `train_dataloader` to estimate number of stepping batches.



DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


                                                                                          
                                                                     
────────────────────────────────────────────────────────
Epoch 0 | Val
────────────────────────────────────────────────────────
Object          Loss Contrastive  Denois.Cat  Denois.Con
────────────────────────────────────────────────────────
jets          3.5110      5.5368           —      0.9448
tracks        5.6870      7.4375      5.1266      0.9958
────────────────────────────────────────────────────────
TOTAL         9.1980
────────────────────────────────────────────────────────

                                                                                                                                  
────────────────────────────────────────────────────────
Epoch 0 | Train
────────────────────────────────────────────────────────
Object          Loss Contrastive  Denois.Cat  Denois.Con
─────────────────────────────

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 200/200 [00:36<00:00,  5.41it/s, v_num=1, train_loss_step=9.200, val_loss=9.200, train_loss_epoch=9.330]
FIT Profiler Report

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Action                                                       	|  Mean duration (s)	|  Num calls      	|  Total time (s) 	|  Percentage %   	|
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Total                                                        	|  -              	|  5170           	|  40.543         	|  100 %          	|
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  run_training_epoch                          

---
## 3 — Lightning AdvancedProfiler (cProfile)

Function-level call-graph inside each Lightning hook.  
Use this when SimpleProfiler identifies a specific hook you want to drill into.

**Overhead:** 5–10× slower than SimpleProfiler — keep step count low.

In [12]:
from lightning.pytorch.profilers import AdvancedProfiler

LIMIT_STEPS_ADV = 100

adv_profiler = AdvancedProfiler(
    dirpath=str(PROF_DIR),
    filename='advanced',
    line_count_restriction=30,   # top-30 functions per hook
)

trainer_adv = L.Trainer(
    **cfg_simple_dict,
    limit_train_batches=LIMIT_STEPS_ADV,
    limit_val_batches=0,
    callbacks=[TQDMProgressBar(refresh_rate=25)],
    logger=CSVLogger('../outputs', name='profiling_advanced'),
    profiler=adv_profiler,
    enable_model_summary=False,
    enable_checkpointing=False,
)

module_adv = build_pretrain_module()
dm_adv = PT_FT_DataModule(cfg, phase='pretrain')

trainer_adv.fit(module_adv, dm_adv)
print(adv_profiler.summary())

/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /storage3/DSIP/rriva/research/fm4tag/.venv/lib/pytho ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
Loading `train_dataloader` to estimate number of stepping batches.



DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


                                                                                                                  
────────────────────────────────────────────────────────
Epoch 0 | Train
────────────────────────────────────────────────────────
Object          Loss Contrastive  Denois.Cat  Denois.Con
────────────────────────────────────────────────────────
jets          3.5248      5.5420           —      0.9978
tracks        5.8580      7.4415      5.9495      1.0159
────────────────────────────────────────────────────────
TOTAL         9.3828
────────────────────────────────────────────────────────

Epoch 0: 100%|██████████| 100/100 [00:18<00:00,  5.28it/s, v_num=0, train_loss_step=9.220, train_loss_epoch=9.380]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 100/100 [00:18<00:00,  5.28it/s, v_num=0, train_loss_step=9.220, train_loss_epoch=9.380]
FIT Profiler Report


In [14]:
# ── Parse saved cProfile files ────────────────────────────────────────────────
# AdvancedProfiler may write .prof binary files alongside the .txt summary.
# We load them with pstats and sort by cumulative time.

prof_files = sorted(PROF_DIR.glob('*.prof'))
if prof_files:
    for pf in prof_files:
        print(f'\n=== {pf.name} (top 20 by cumulative time) ===')
        buf = io.StringIO()
        pstats.Stats(str(pf), stream=buf).sort_stats('cumulative').print_stats(20)
        print(buf.getvalue())
else:
    print('No .prof files found — showing .txt summary.')
    for tf in sorted(PROF_DIR.glob('advanced*.txt')):
        print(f'\n=== {tf.name} ===')
        print(tf.read_text()[:6000])

No .prof files found — showing .txt summary.


---
## 4 — Lightning PyTorchProfiler

GPU + CPU operator-level traces via `torch.profiler`.  
Writes a Chrome trace JSON — open it at **https://ui.perfetto.dev/** for a flame chart.

We profile a small number of steps to keep file sizes manageable.

In [16]:
from lightning.pytorch.profilers import PyTorchProfiler

LIMIT_STEPS_PT = 50

pt_profiler = PyTorchProfiler(
    dirpath=str(PROF_DIR),
    filename='pytorch_trace',
    export_to_chrome=True,   # writes .json for chrome://tracing / Perfetto
    with_stack=True,         # call-stack annotation in the trace
    profile_memory=True,     # GPU/CPU memory allocation per op
    row_limit=30,
)

trainer_pt = L.Trainer(
    **cfg_simple_dict,
    limit_train_batches=LIMIT_STEPS_PT,
    limit_val_batches=0,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=CSVLogger('../outputs', name='profiling_pytorch'),
    profiler=pt_profiler,
    enable_model_summary=False,
    enable_checkpointing=False,
)

module_pt = build_pretrain_module()
dm_pt = PT_FT_DataModule(cfg, phase='pretrain')

trainer_pt.fit(module_pt, dm_pt)
print(pt_profiler.summary())

traces = sorted(PROF_DIR.glob('*.json'))
print('\nTrace files (open in https://ui.perfetto.dev/):')
for t in traces:
    print(f'  {t}')

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
Loading `train_dataloader` to estimate number of stepping batches.



DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000


/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (50) is smaller than the logging interval Trainer(log_every_n_steps=100). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0:   0%|          | 0/50 [00:00<?, ?it/s]

/storage3/DSIP/rriva/research/fm4tag/.venv/lib/python3.13/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
[W226 10:26:07.187619418 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


                                                                                                                
────────────────────────────────────────────────────────
Epoch 0 | Train
────────────────────────────────────────────────────────
Object          Loss Contrastive  Denois.Cat  Denois.Con
────────────────────────────────────────────────────────
jets          3.5271      5.5435           —      1.0047
tracks        5.9462      7.4374      6.4093      1.0095
────────────────────────────────────────────────────────
TOTAL         9.4733
────────────────────────────────────────────────────────

Epoch 0: 100%|██████████| 50/50 [00:14<00:00,  3.36it/s, v_num=0, train_loss_step=9.280, train_loss_epoch=9.470]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 50/50 [00:14<00:00,  3.35it/s, v_num=0, train_loss_step=9.280, train_loss_epoch=9.470]
FIT Profiler Report
Profile stats for: records
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       

---
## 5 — Manual Micro-benchmarks

Precise GPU timing of **individual sub-components** using `torch.cuda.Event`.  
Unlike wall-clock (`time.perf_counter`), CUDA events account for async kernel scheduling.

We call `module._compute_loss()` directly (bypassing `training_step` / Lightning logging)
and also time individual sub-steps:
* `embed_data` — embedding lookup + continuous MLP
* `encoder.forward` — SAINT transformer
* full forward + loss
* full forward + backward

In [17]:
# ── Build a standalone module and grab a real batch ───────────────────────────
module_mb = build_pretrain_module()

dm_mb = PT_FT_DataModule(cfg, phase='pretrain')
dm_mb.setup('fit')
dl_mb = dm_mb.train_dataloader()
batch_mb = to_device(next(iter(dl_mb)), DEVICE_STR)

print(f'Batch size : {batch_mb["label"].shape[0]}')
print(f'Device     : {DEVICE_STR}')


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/val/output/pp_output_val.h5
  samples : 100000
Batch size : 256
Device     : cuda:0


In [18]:
def cuda_time_ms(fn, n_repeats: int = 20, warmup: int = 5) -> float:
    """Return median execution time of *fn()* in milliseconds.

    Uses ``torch.cuda.Event`` for GPU-correct timing when CUDA is available,
    otherwise falls back to ``time.perf_counter``.
    """
    if not torch.cuda.is_available():
        times = []
        for i in range(warmup + n_repeats):
            t0 = time.perf_counter()
            fn()
            if i >= warmup:
                times.append((time.perf_counter() - t0) * 1e3)
        return sorted(times)[len(times) // 2]

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    times = []
    for i in range(warmup + n_repeats):
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        if i >= warmup:
            times.append(start.elapsed_time(end))
    return sorted(times)[len(times) // 2]  # median


print('cuda_time_ms defined.')

cuda_time_ms defined.


In [19]:
# ── embed_data ────────────────────────────────────────────────────────────────
obj = cfg.constituent_objects[0]   # 'tracks'
enc = module_mb.encoders[obj]
feats = batch_mb['constituents'][obj]

# Replicate the valid-mask flattening done inside PretrainModule
from einops import rearrange
valid_flat = rearrange(feats['valid'],       'b c -> (b c)')
cat_flat   = rearrange(feats['categorical'], 'b c f -> (b c) f')[valid_flat]
con_flat   = rearrange(feats['continuous'],  'b c f -> (b c) f')[valid_flat]
print(f'N_valid = {valid_flat.sum().item()}')

def bench_embed():
    with torch.no_grad():
        embed_data(cat_flat, con_flat, enc)

t_embed = cuda_time_ms(bench_embed)
print(f'embed_data      : {t_embed:>8.2f} ms')


# ── encoder.forward ───────────────────────────────────────────────────────────
with torch.no_grad():
    cat_emb, con_emb = embed_data(cat_flat, con_flat, enc)

def bench_encoder():
    with torch.no_grad():
        enc(cat_emb, con_emb)

t_enc = cuda_time_ms(bench_encoder)
print(f'encoder.forward : {t_enc:>8.2f} ms')

N_valid = 1759
embed_data      :     0.77 ms
encoder.forward :    26.72 ms


In [20]:
# ── Full forward pass (all objects, all losses) ───────────────────────────────
# We call _compute_loss() directly — no Lightning logging context needed.

module_mb.eval()

def bench_forward():
    with torch.no_grad():
        forward_loss(module_mb, batch_mb)

t_fwd = cuda_time_ms(bench_forward)
print(f'full forward + loss : {t_fwd:>8.2f} ms')


# ── Forward + Backward ────────────────────────────────────────────────────────
module_mb.train()
opt_mb = torch.optim.AdamW(module_mb.parameters(), lr=1e-4)

def bench_fwd_bwd():
    opt_mb.zero_grad(set_to_none=True)
    loss = forward_loss(module_mb, batch_mb)
    loss.backward()
    opt_mb.step()

t_fwd_bwd = cuda_time_ms(bench_fwd_bwd, n_repeats=10)
print(f'forward + backward  : {t_fwd_bwd:>8.2f} ms')

bwd_overhead = t_fwd_bwd - t_fwd
print(f'\nBackward overhead   : {bwd_overhead:>8.2f} ms  '
      f'({100 * bwd_overhead / t_fwd_bwd:.1f}% of fwd+bwd)')

full forward + loss :    52.04 ms
forward + backward  :   186.27 ms

Backward overhead   :   134.22 ms  (72.1% of fwd+bwd)


In [ ]:
# ── Batch-size scaling ────────────────────────────────────────────────────────
# Does forward throughput scale linearly with batch size?
# Sub-linear scaling often points to load imbalance across valid tokens.

print(f'{"B":>6}  {"time (ms)":>10}  {"samples/s":>12}  {"ms/sample":>10}')
print('-' * 45)

bs_results: dict[int, tuple[float, float]] = {}   # bs → (time_ms, sps)

for bs in [64, 80, 96, 112, 128, 144, 160, 176, 192, 208, 224, 240, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448, 464, 480, 496, 512, 1024]:
    c_bs = make_cfg(**{'pretrain_dataloader.batch_size': bs})
    dm_bs = PT_FT_DataModule(c_bs, phase='pretrain')
    dm_bs.setup('fit')
    raw_bs = next(iter(dm_bs.train_dataloader()))
    b_bs = to_device(raw_bs, DEVICE_STR)

    mod_bs = build_pretrain_module(c_bs)
    mod_bs.eval()

    def _fn_bs():
        with torch.no_grad():
            forward_loss(mod_bs, b_bs)

    try:
        t = cuda_time_ms(_fn_bs, n_repeats=10, warmup=3)
        sps = bs / (t / 1e3)
        bs_results[bs] = (t, sps)
        print(f'{bs:>6}  {t:>10.2f}  {sps:>12,.0f}  {t/bs:>10.4f}')
    except torch.cuda.OutOfMemoryError:
        print(f'{bs:>6}  OOM')
        torch.cuda.empty_cache()

In [ ]:
if bs_results:
    import matplotlib.pyplot as plt

    batches = sorted(bs_results)
    times   = [bs_results[b][0] for b in batches]
    sps     = [bs_results[b][1] for b in batches]
    mspersample = [t / b for b, t in zip(batches, times)]
    best_bs = max(bs_results, key=lambda b: bs_results[b][1])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # ── left: throughput ──────────────────────────────────────────────────────
    ax1.plot(batches, sps, marker='o', linewidth=2)
    ax1.axvline(best_bs, color='red', linestyle='--', alpha=0.7,
                label=f'best B={best_bs} ({bs_results[best_bs][1]:,.0f} sps)')
    ax1.set_xlabel('Batch size')
    ax1.set_ylabel('Throughput (samples/s)')
    ax1.set_title('Forward throughput vs batch size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # ── right: ms/sample ──────────────────────────────────────────────────────
    ax2.plot(batches, mspersample, marker='o', color='C1', linewidth=2)
    ax2.axvline(best_bs, color='red', linestyle='--', alpha=0.7,
                label=f'best B={best_bs}')
    ax2.set_xlabel('Batch size')
    ax2.set_ylabel('ms / sample')
    ax2.set_title('Per-sample latency vs batch size')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Batch-size scaling (forward pass, no grad)', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 6 — `torch.profiler` Deep Dive

Direct use of `torch.profiler.profile` — more control than the Lightning wrapper.

**Schedule:** `wait=1` (skip) → `warmup=3` (warm CUDA) → `active=5` (record).  
Sorting views:
* `self_cuda_time_total` — actual GPU kernel time (excludes child calls)
* `self_cpu_time_total` — CPU dispatch overhead
* `cuda_memory_usage` — GPU memory per op
* `group_by_input_shape` — spots padded/wasted compute

In [ ]:
import torch.profiler as torch_prof

TRACE_DIR = PROF_DIR / 'torch_prof_trace'
TRACE_DIR.mkdir(parents=True, exist_ok=True)

module_dp = build_pretrain_module()
module_dp.train()
opt_dp = torch.optim.AdamW(module_dp.parameters(), lr=1e-4)

# wait=1 step skipped, warmup=3 no-record, active=5 recorded
schedule = torch_prof.schedule(wait=1, warmup=3, active=5, repeat=1)

activities = [torch_prof.ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(torch_prof.ProfilerActivity.CUDA)

it_dp = iter(dl_mb)

def _next_batch():
    global it_dp
    try:
        return to_device(next(it_dp), DEVICE_STR)
    except StopIteration:
        it_dp = iter(dl_mb)
        return to_device(next(it_dp), DEVICE_STR)


with torch_prof.profile(
    activities=activities,
    schedule=schedule,
    on_trace_ready=torch_prof.tensorboard_trace_handler(str(TRACE_DIR)),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step in range(10):  # wait(1) + warmup(3) + active(5) + 1 extra
        b = _next_batch()
        opt_dp.zero_grad(set_to_none=True)
        loss = forward_loss(module_dp, b)
        loss.backward()
        opt_dp.step()
        prof.step()

print(f'Trace written to: {TRACE_DIR}')
print('Open it at https://ui.perfetto.dev/')

In [ ]:
print('=== Top 20 ops by self CUDA time ===')
print(prof.key_averages().table(sort_by='self_cuda_time_total', row_limit=20))

In [ ]:
print('=== Top 20 ops by self CPU time ===')
print(prof.key_averages().table(sort_by='self_cpu_time_total', row_limit=20))

In [ ]:
if torch.cuda.is_available():
    print('=== Top 20 ops by CUDA memory usage ===')
    print(prof.key_averages().table(sort_by='cuda_memory_usage', row_limit=20))

In [ ]:
# ── Group by input shape — reveals padding / variable-length waste ─────────────
print('=== Top 15 ops grouped by input shapes ===')
print(prof.key_averages(group_by_input_shape=True).table(
    sort_by='self_cuda_time_total', row_limit=15
))

---
## 7 — GPU Memory Audit

Track peak VRAM, per-step allocations, and identify memory spikes.  
Important before scaling batch size or switching precision.

In [ ]:
peak_alloc = peak_reserved = total_vram = 0.0
mem_alloc = mem_reserved = []

if not torch.cuda.is_available():
    print('CUDA not available — skipping memory audit.')
else:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    module_mem = build_pretrain_module()
    module_mem.train()
    opt_mem = torch.optim.AdamW(module_mem.parameters(), lr=1e-4)

    mem_alloc, mem_reserved = [], []
    it_mem = iter(dl_mb)

    for step in range(20):
        try:
            raw = next(it_mem)
        except StopIteration:
            it_mem = iter(dl_mb)
            raw = next(it_mem)

        b = to_device(raw, DEVICE_STR)
        opt_mem.zero_grad(set_to_none=True)
        loss = forward_loss(module_mem, b)
        loss.backward()
        opt_mem.step()
        torch.cuda.synchronize()

        mem_alloc.append(torch.cuda.memory_allocated() / 1e9)
        mem_reserved.append(torch.cuda.memory_reserved() / 1e9)

    peak_alloc    = torch.cuda.max_memory_allocated() / 1e9
    peak_reserved = torch.cuda.max_memory_reserved()  / 1e9
    total_vram    = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f'Peak allocated  : {peak_alloc:.2f} GB  '
          f'({100 * peak_alloc / total_vram:.1f}% of {total_vram:.0f} GB)')
    print(f'Peak reserved   : {peak_reserved:.2f} GB')
    print(f'Step mean alloc : {sum(mem_alloc) / len(mem_alloc):.2f} GB')
    print(f'Step max  alloc : {max(mem_alloc):.2f} GB')
    print(f'Step min  alloc : {min(mem_alloc):.2f} GB')

In [ ]:
if torch.cuda.is_available() and mem_alloc:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 3))
        steps = list(range(len(mem_alloc)))
        ax.plot(steps, mem_alloc,    label='Allocated', linewidth=2)
        ax.plot(steps, mem_reserved, label='Reserved',  linewidth=2, linestyle='--')
        ax.axhline(total_vram, color='red', linestyle=':', label='Total VRAM')
        ax.set_xlabel('Training step')
        ax.set_ylabel('GPU memory (GB)')
        ax.set_title('GPU memory per training step')
        ax.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('matplotlib not installed — skipping plot.')

In [ ]:
# ── bf16-mixed vs float32 memory comparison ───────────────────────────────────
if torch.cuda.is_available():
    print(f'{"Precision":<14}  {"Peak VRAM (GB)":>14}  {"% of total":>10}')
    print('-' * 45)
    for precision in ['bf16-mixed', '32-true']:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

        c_prec = make_cfg(**{'trainer.precision': precision})

        tr_prec = L.Trainer(
            **OmegaConf.to_container(c_prec.trainer, resolve=True),
            limit_train_batches=10,
            limit_val_batches=0,
            logger=False,
            enable_checkpointing=False,
            enable_model_summary=False,
            enable_progress_bar=False,
        )
        mod_prec = build_pretrain_module(c_prec)
        dm_prec  = PT_FT_DataModule(c_prec, phase='pretrain')
        tr_prec.fit(mod_prec, dm_prec)

        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f'{precision:<14}  {peak:>14.2f}  {100*peak/total_vram:>9.1f}%')

---
## 8 — Bottleneck Summary

Collect all findings into one actionable table.

In [ ]:
print('=' * 70)
print('PROFILING SUMMARY')
print('=' * 70)

print('\n── 1. Dataloader ──')
print(f'  Config default           : {thr_cpu:>10,.0f} samples/s (CPU-only)')
if thr_gpu:
    print(f'  Config default           : {thr_gpu:>10,.0f} samples/s (incl. H→D)')
print(f'  Best num_workers = {best_nw:>2d}    : {results_workers[best_nw]:>10,.0f} samples/s')
print(f'  Best prefetch    = {best_pf!s:<4}   : {results_prefetch[best_pf]:>10,.0f} samples/s')

print('\n── 2. Lightning SimpleProfiler ──')
print(f'  Output : {PROF_DIR}/simple.txt')

print('\n── 3. Lightning AdvancedProfiler ──')
print(f'  Output : {PROF_DIR}/advanced.txt')

print('\n── 4. Lightning PyTorchProfiler ──')
print(f'  Trace  : {PROF_DIR}/pytorch_trace.json')
print('  → open at https://ui.perfetto.dev/')

print('\n── 5. Micro-benchmarks ──')
print(f'  embed_data         : {t_embed:>8.2f} ms')
print(f'  encoder.forward    : {t_enc:>8.2f} ms')
print(f'  full forward+loss  : {t_fwd:>8.2f} ms')
print(f'  forward + backward : {t_fwd_bwd:>8.2f} ms')

print('\n── 6. torch.profiler deep dive ──')
print(f'  Trace  : {TRACE_DIR}/')
print('  → open in TensorBoard (tensorboard --logdir) or Perfetto')

if torch.cuda.is_available() and mem_alloc:
    print('\n── 7. GPU Memory ──')
    print(f'  Peak allocated : {peak_alloc:.2f} GB  '
          f'({100 * peak_alloc / total_vram:.1f}% of {total_vram:.0f} GB)')
    print(f'  Peak reserved  : {peak_reserved:.2f} GB')

print('\n' + '=' * 70)